# image_qwen

**Qwen-Image** (Alibaba, 20B) 로 World Reference → 장소 배경 → 인물 초상.

| | |
|---|---|
| 커널 | `qwen` |
| 출력 | `outputs/image_qwen/<시나리오>/` |

---

## 왜 이 모델인가

Apache 2.0, 게이트 없음. diffusers 에 `QwenImagePipeline` 이 이미 있다(아직
정식 릴리스 전이라 git 최신판 필요 — `setup_kernels.sh --only qwen` 가 처리함).
BF16 그대로면 ~48GB 로 이 GPU 에 거의 꽉 차 위험해서, 트랜스포머만
bitsandbytes 4bit 로 양자화해 ~17-18GB 로 낮췄다.

**klein/HiDream-Dev 와 달리 이 모델은 `negative_prompt` + `true_cfg_scale` 로
진짜 classifier-free guidance 가 작동한다** — 장소 이미지에 사람이 새는 문제를
프롬프트 강화뿐 아니라 negative_prompt 로도 직접 막을 수 있다.


## 1. 커널 확인


In [ ]:
import sys, importlib

KERNEL = "qwen"
NEED   = ["torch", "diffusers", "transformers", "bitsandbytes", "PIL"]

print("python     :", sys.version.split()[0])
print("interpreter:", sys.executable)
print()

missing = []
for m in NEED:
    try:
        mod = importlib.import_module(m)
        print(f"OK   {m:16s} {getattr(mod, '__version__', '')}")
    except Exception as e:
        missing.append(m)
        print(f"FAIL {m:16s} {type(e).__name__}: {e}")

assert not missing, (
    f"\n{missing} 를 import 하지 못했다.\n"
    f"커널을 '{KERNEL}' 로 바꿀 것 (Kernel > Change kernel).\n"
    f"'{KERNEL}' 커널이 목록에 없으면 setup_kernels.sh --only qwen 을 먼저 실행한다."
)

from diffusers import QwenImagePipeline, QwenImageTransformer2DModel, BitsAndBytesConfig
print("\nQwenImagePipeline import OK")


## 2. 인증
Apache 2.0, 게이트 없다.


In [ ]:
import os, pathlib
tok = os.environ.get("HF_TOKEN", "").strip()
if not tok:
    p = pathlib.Path.home() / ".hf_token"
    if p.exists():
        tok = p.read_text().strip()
        os.environ["HF_TOKEN"] = tok
print("토큰:", (tok[:6] + "…") if tok else "(없음 — 게이트 없는 모델이라 없어도 된다)")


## 3. GPU 확인


In [ ]:
import torch

assert torch.cuda.is_available(), "GPU가 없다. 계산 노드에서 실행할 것."
p = torch.cuda.get_device_properties(0)
print(f"GPU  : {p.name}")
print(f"VRAM : {round(p.total_memory / 1e9, 1)} GB")
print(f"CC   : {p.major}.{p.minor}")
print("\n트랜스포머 4bit 양자화 기준 대략 17-18GB 필요.")


## 4. 설정


In [ ]:
import os, sys, json, pathlib, getpass, shutil

USER = getpass.getuser()

def pick_root():
    for c in (f"/mnt/data1/{USER}", f"/data1/{USER}",
              str(pathlib.Path.home()), "/workspace", "/scratch"):
        if pathlib.Path(c).is_dir() and os.access(c, os.W_OK):
            return c
    return None

ROOT = os.environ.get("PROJ_ROOT") or pick_root()
if ROOT is None:
    raise SystemExit(f"쓸 수 있는 루트를 못 찾았다 (user={USER})")

if not pathlib.Path(os.environ.get("HOME", "/nonexistent")).is_dir():
    os.environ["HOME"] = ROOT

BASE  = pathlib.Path(os.environ.get("PROJ_BASE") or f"{ROOT}/mystery")
MODEL = "image_qwen_edit"
OUT_DIR = BASE / "outputs" / MODEL

os.environ.setdefault("HF_HOME",      str(BASE / "hf_cache"))
os.environ.setdefault("TMPDIR",       str(BASE / "tmp"))
os.environ.setdefault("MPLCONFIGDIR", str(BASE / "mpl_cache"))

for p in (OUT_DIR, pathlib.Path(os.environ["HF_HOME"]), pathlib.Path(os.environ["TMPDIR"])):
    p.mkdir(parents=True, exist_ok=True)

# ---- 입력 JSON (scenarios_en/*.json 전부) ---------------------------------
def pick_input_dir():
    cands = []
    if os.environ.get("PROJ_INPUT"):
        cands.append(pathlib.Path(os.environ["PROJ_INPUT"]))
    cands += [pathlib.Path.cwd(), pathlib.Path.cwd().parent, BASE]
    for c in cands:
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    for c in pathlib.Path(ROOT).glob("*/*/"):
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    return None

IN_DIR = pick_input_dir()
if IN_DIR is None:
    raise SystemExit(
        "scenarios_en/*.json 을 못 찾았다.\n"
        f"  이 셀 맨 위에서:  os.environ['PROJ_INPUT'] = '/data1/{USER}/<저장소>/Jaemin'"
    )

SCENARIO_FILES = sorted((IN_DIR / "scenarios_en").glob("*.json"))
assert SCENARIO_FILES, f"scenarios_en 에 json 이 없다: {IN_DIR / 'scenarios_en'}"

DATA = {}
for _f in SCENARIO_FILES:
    _d = json.load(open(_f, encoding="utf-8"))
    DATA[_d["scenario_id"]] = _d

for k in DATA:
    (OUT_DIR / k).mkdir(parents=True, exist_ok=True)


def out_path(scen, *parts):
    p = OUT_DIR / scen
    for x in parts[:-1]:
        p = p / x
    p.mkdir(parents=True, exist_ok=True)
    return p / parts[-1]


print("node    :", os.uname().nodename)
print("BASE    :", BASE, f"(여유 {shutil.disk_usage(BASE).free/1e9:.0f} GB)")
print("IN_DIR  :", IN_DIR)
print("HF_HOME :", os.environ["HF_HOME"])
print("OUT_DIR :", OUT_DIR)
print(f"시나리오 {len(DATA)}개")


## 5. 스타일

In [ ]:
STYLE = ("cohesive painterly realism, muted desaturated palette, "
         "soft directional daylight, film grain, no text, no watermark")


## 6. 모델 로드 (Qwen-Image-Edit)

In [ ]:
import torch
from diffusers import QwenImageEditPipeline, QwenImageTransformer2DModel, BitsAndBytesConfig
from PIL import Image

MODEL_ID = "Qwen/Qwen-Image-Edit"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

transformer = QwenImageTransformer2DModel.from_pretrained(
    MODEL_ID, subfolder="transformer",
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,
)

pipe = QwenImageEditPipeline.from_pretrained(
    MODEL_ID, transformer=transformer, torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()

print("준비됨:", MODEL_ID)
print(f"할당된 VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 7. 생성 함수 (image+text)

In [ ]:
STEPS = 40
TRUE_CFG_SCALE = 4.0


def gen(prompt, ref_image_path, out_file, seed=0, size=1024):
    ref = Image.open(ref_image_path).convert("RGB")
    g = torch.Generator("cuda").manual_seed(seed)
    image = pipe(
        image=ref,
        prompt=prompt,
        negative_prompt=" ",
        true_cfg_scale=TRUE_CFG_SCALE,
        num_inference_steps=STEPS,
        generator=g,
    ).images[0]
    out_file = pathlib.Path(out_file)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    image.save(out_file)
    return out_file


print("준비됨. 다음 셀에서 재연 컷 5장 생성.")


## 8. 재연 컷 설정

In [ ]:
# ---------------------------------------------------------------------------
# 재연 컷 5장 — 12_흥부전 범인(C3, 마름/매향)의 이미 만들어둔 초상을 레퍼런스로
# 넣고, 각 컷의 서사를 텍스트로 지시한다(image + text). 얼굴 일관성 유지가
# 목적 — 이 컷들만 유일하게 범인 얼굴을 공개해도 되는 자리다(팀 문서 명시:
# "게임 중 금지되던 범인 얼굴이 여기서 처음 공개됩니다").
# ---------------------------------------------------------------------------
import json as _json

PILOT_ID = "12_heungbu"
CULPRIT_ID = "C3"  # is_culprit: true (json 확인함)

d = DATA[PILOT_ID]
era, loc = d["setting"]["era"], d["setting"]["location"]

# 레퍼런스: fluxdev 쪽에서 이미 만든 "calm" 표정 초상을 기준 얼굴로 쓴다.
REF_IMAGE = OUT_DIR.parent / "image_fluxdev" / PILOT_ID / "portraits_emotion" / f"{CULPRIT_ID}_calm.png"
assert REF_IMAGE.exists(), f"레퍼런스 초상이 없다: {REF_IMAGE} (fluxdev 확장 작업을 먼저 돌릴 것)"

REENACT_CUTS = [
    ("cut1_that_night",
     "the same person's face, now hardening with resolve in dim lamplight — "
     "if the truth about the storehouse ledger comes out, she'll be turned "
     "over to the authorities. close-up on her face, night setting, tense "
     "expression."),
    ("cut2_approach",
     "the same person seen from behind, walking quietly down a dim hallway "
     "toward a reception room at night, unnoticed by anyone, tense posture, "
     "candle-lit corridor."),
    ("cut3_the_act",
     "the same person's silhouette in a dim reception room at night, "
     "reaching toward a bowl of medicinal tonic on a low desk, implied "
     "tension rather than graphic action, dark and shadowy composition."),
    ("cut4_cover_up",
     "the same person's face contorting into feigned anguish and grief, "
     "false tears, performing distress for onlookers, dim interior "
     "setting."),
    ("cut5_next_morning",
     "the same person standing calm and composed in daylight outside a "
     "quiet reception room at dawn, deceptively serene expression, soft "
     "morning light, no one else around."),
]

def reenact_prompt(desc):
    return (f"{desc} setting: {era}, {loc}. mood: tense, quietly dreadful. "
            f"{STYLE}. keep the same facial identity, hairstyle, and "
            f"clothing as the reference image.")

print(f"재연 컷 {len(REENACT_CUTS)}개, 레퍼런스: {REF_IMAGE}")


## 9. 생성

In [ ]:
EXT_OUT = OUT_DIR / PILOT_ID / "reenact_cuts"
EXT_OUT.mkdir(parents=True, exist_ok=True)

for cid, desc in REENACT_CUTS:
    out_f = EXT_OUT / f"{cid}.png"
    if out_f.exists():
        print("   (이미 있음, 건너뜀)", cid); continue
    gen(reenact_prompt(desc), REF_IMAGE, out_f, seed=0)
    print("OK reenact", cid)

print("\n저장 위치:", EXT_OUT)
